# Import International Monetary Fund (IMF) data
- Author: Bryan Bravo
- Created: 2026-03-19
## Import Libraries

In [1]:
import os
import sys
os.chdir("C:/Users/bravo/OneDrive/bravo_projects/MLProject/StraitofHormuzResearch")


# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

import sdmx

# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


### Custom Functions

### Variables

In [3]:
end_date = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")

## Query

In [4]:
country_mapping = {
    'australia': 'AUS',
    'brazil': 'BRA',
    'canada': 'CAN',
    'china': 'CHN',
    # 'euro': 'EU',  # No IMF Data, must source from elsewhere.
    'france': 'FRA',
    'germany': 'DEU',
    'india': 'IND',
    'italy': 'ITA',
    'japan': 'JPN',
    'mexico': 'MEX',
    'south_korea': 'KOR',
    'russia': 'RUS',
    'south_africa': 'ZAF',
    'turkiye': 'TUR',
    'united_kingdom': 'GBR',
    'united_states': 'USA'
}
countries = [country for country in country_mapping.values()]


In [5]:

key = f"{'+'.join(countries)}.CPI._T.IX.M"


print(f"Requesting data for key: {key} starting {2006}...")
IMF_DATA = sdmx.Client('IMF_DATA')
try:
    data_msg = IMF_DATA.data('CPI', key=key, params={'startPeriod': 2006, 'endPeriod': end_date[0:4]})
except Exception as e:
    print(f"An error occurred: {e}")
    exit()

cpi_df = sdmx.to_pandas(data_msg).reset_index()


cpi_df.columns = [col.lower() for col in cpi_df.columns]
# Subset columns for use
cpi_df = cpi_df[['time_period', 'country', 'value']]

# Extract year and Month
cpi_df[['year', 'month']] = cpi_df['time_period'].str.split('-', expand=True)
cpi_df['year'] = cpi_df['year'].astype(int)
cpi_df['month'] = cpi_df['month'].str[1:].astype(int)


# Remap countries
cpi_df['country'] = cpi_df['country'].map({
    code: cntry for cntry, code in country_mapping.items()
})

cpi_df.drop('time_period', axis=1, inplace=True)

# Convert to Spark DF
cpi_df = spark.createDataFrame(cpi_df)
cpi_df.cache().count()


Requesting data for key: AUS+BRA+CAN+CHN+FRA+DEU+IND+ITA+JPN+MEX+KOR+RUS+ZAF+TUR+GBR+USA.CPI._T.IX.M starting 2006...


xml.Reader got no structure=… argument for StructureSpecificData


3645

In [6]:
print(cpi_df.dtypes, "\n(Row, Col) Count: ", (cpi_df.count(), len(cpi_df.columns)))
cpi_df.show(5)

[('country', 'string'), ('value', 'double'), ('year', 'bigint'), ('month', 'bigint')] 
(Row, Col) Count:  (3645, 4)
+---------+-----+----+-----+
|  country|value|year|month|
+---------+-----+----+-----+
|australia| 96.4|2024|    4|
|australia|96.17|2024|    5|
|australia|96.52|2024|    6|
|australia|96.77|2024|    7|
|australia|96.49|2024|    8|
+---------+-----+----+-----+
only showing top 5 rows

